In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part2c_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
districts = pd.read_csv(RAW_DIR / "districts.csv")
age_sex = pd.read_csv(RAW_DIR / "age_sex.csv")
scenarios = pd.read_csv(RAW_DIR / "projection_scenarios.csv")

proj_district = "D05"
d_row = districts[districts["district_id"] == proj_district].iloc[0]
sub = age_sex[age_sex["district_id"] == proj_district].copy()
ages = sorted(sub["age_start"].unique().tolist())

# 2a totals
p_2020 = float(d_row["population_2020"])
p_2025 = float(sub["population_2025"].sum())
b_annual = (p_2025 - p_2020) / 5.0
p_2030_arith = p_2025 + 5.0 * b_annual
g_annual = (p_2025 / p_2020) ** (1.0 / 5.0)
p_2030_geom = p_2025 * (g_annual ** 5.0)

pop25 = {(r["age_start"], r["sex"]): float(r["population_2025"]) for _, r in sub.iterrows()}
surv = {(r["age_start"], r["sex"]): float(r["survival_5yr"]) for _, r in sub.iterrows()}
arr = {(r["age_start"], r["sex"]): float(r["arrivals_2025"]) for _, r in sub.iterrows()}
dep = {(r["age_start"], r["sex"]): float(r["departures_2025"]) for _, r in sub.iterrows()}
births_2025 = float(sub[sub["sex"] == "F"]["births_to_women_2025"].sum())

# ---------- Cohort-component projection ----------
def project_cohort_component(fert_mult, arr_mult, dep_mult, nb_surv_f, nb_surv_m):
    births_5yr = 5.0 * births_2025 * fert_mult
    births_sex = {"F": 0.49 * births_5yr, "M": 0.51 * births_5yr}
    nb_surv = {"F": nb_surv_f, "M": nb_surv_m}

    pre_mig = {}
    pop2030 = {}
    for sex in ["F", "M"]:
        pre_mig[(0, sex)] = births_sex[sex] * nb_surv[sex]
        for idx in range(1, len(ages) - 1):
            a = ages[idx]
            prev_a = ages[idx - 1]
            pre_mig[(a, sex)] = pop25[(prev_a, sex)] * surv[(prev_a, sex)]
        pre_mig[(85, sex)] = pop25[(80, sex)] * surv[(80, sex)] + pop25[(85, sex)] * surv[(85, sex)]

        for a in ages:
            nm_5yr = 5.0 * (arr[(a, sex)] * arr_mult - dep[(a, sex)] * dep_mult)
            pop2030[(a, sex)] = pre_mig[(a, sex)] + nm_5yr

    tot = sum(pop2030.values())
    u15 = sum(pop2030[(a, s)] for a in ages if a < 15 for s in ["F", "M"])
    o65 = sum(pop2030[(a, s)] for a in ages if a >= 65 for s in ["F", "M"])
    a0_4 = sum(pop2030[(0, s)] for s in ["F", "M"])
    return {
        "births_5yr": births_5yr,
        "pop_2030": tot,
        "share_under_15_pct": (u15 / tot) * 100.0,
        "share_65_plus_pct": (o65 / tot) * 100.0,
        "pop_0_4_2030": a0_4,
        "pop_0_4_f": pop2030[(0, "F")],
        "pop_0_4_m": pop2030[(0, "M")],
    }

cc_by_name = {
    sc["scenario"]: project_cohort_component(
        float(sc["fertility_multiplier"]),
        float(sc["arrivals_multiplier"]),
        float(sc["departures_multiplier"]),
        float(sc["newborn_survival_f"]),
        float(sc["newborn_survival_m"]),
    )
    for _, sc in scenarios.iterrows()
}

# ---------- Model comparison ----------
comparison_rows = [
    {"model": "Arithmetic (2020-2025)", "pop_2030": p_2030_arith, "share_under_15_pct": np.nan, "share_65_plus_pct": np.nan},
    {"model": "Geometric (2020-2025)", "pop_2030": p_2030_geom, "share_under_15_pct": np.nan, "share_65_plus_pct": np.nan},
]
for sc_name in ["Low", "Base", "High"]:
    r = cc_by_name[sc_name]
    comparison_rows.append({
        "model": f"Cohort-Component ({sc_name})",
        "pop_2030": r["pop_2030"],
        "share_under_15_pct": r["share_under_15_pct"],
        "share_65_plus_pct": r["share_65_plus_pct"],
    })

comp_df = pd.DataFrame(comparison_rows)
print("\nModel comparison:")
display(comp_df)
safe_save(lambda p: comp_df.to_csv(p, index=False), "part2c_model_comparison_d05.csv")

# ---------- Fertility sensitivity (Base scenario, fertility set to High) ----------
base_sc = scenarios[scenarios["scenario"] == "Base"].iloc[0]
high_sc = scenarios[scenarios["scenario"] == "High"].iloc[0]
sens_only_fert = project_cohort_component(
    float(high_sc["fertility_multiplier"]),
    float(base_sc["arrivals_multiplier"]),
    float(base_sc["departures_multiplier"]),
    float(base_sc["newborn_survival_f"]),
    float(base_sc["newborn_survival_m"]),
)

diff_0_4 = sens_only_fert["pop_0_4_2030"] - cc_by_name["Base"]["pop_0_4_2030"]
pct_0_4 = (diff_0_4 / cc_by_name["Base"]["pop_0_4_2030"]) * 100.0

sens_df = pd.DataFrame([{
    "base_0_4_2030": cc_by_name["Base"]["pop_0_4_2030"],
    "high_fert_only_0_4_2030": sens_only_fert["pop_0_4_2030"],
    "abs_change_0_4": diff_0_4,
    "pct_change_0_4": pct_0_4,
}])

print("\nFertility sensitivity (ages 0-4):")
display(sens_df)
safe_save(lambda p: sens_df.to_csv(p, index=False), "part2c_fertility_sensitivity_d05.csv")

BASE_DIR: C:\Users\takudzwa\Documents\Group-4-Assignment-2-Two-

Model comparison:


,model,pop_2030,share_under_15_pct,share_65_plus_pct
0,Arithmetic (2020-2025),167030.000000,NaN,NaN
1,Geometric (2020-2025),168573.264780,NaN,NaN
2,Cohort-Component (Low),166441.244099,16.137949,13.746511
3,Cohort-Component (Base),173151.085551,17.114559,13.519617
4,Cohort-Component (High),179921.828767,18.046052,13.305147


Saved: C:\Users\takudzwa\Documents\Group-4-Assignment-2-Two-\outputs\part2c_model_comparison_d05.csv

Fertility sensitivity (ages 0-4):


,base_0_4_2030,high_fert_only_0_4_2030,abs_change_0_4,pct_change_0_4
0,17442.6424,19469.759488,2027.117088,11.621617


Saved: C:\Users\takudzwa\Documents\Group-4-Assignment-2-Two-\outputs\part2c_fertility_sensitivity_d05.csv
